In [ ]:
'''
This module uses multiple pyod models as an unsupervised machine learning algorithms. 
Using a majority vote of the 3 most different paramameterized models of the same type, e.g. 6 definitions of IForest, keep the 3 most different.
The models use a preprocessed dataframe, with all answers, provided by the specific dataframeBuilder for the test
Additionaly, the models are also fitted on a reduced dimension, using Principal Component Analysis.
Each row represents a test instance.

Input:
  - ./decoded_data/{TESTNAME}/FactQuestion{TESTNAME}.csv
  - ./decoded_data/{TESTNAME}/FactTest.csv
  - ./decoded_data/DimCandidate.csv

Output:
  - ./anomalies/{TESTNAME}/csv/pyod_ensemble.csv
'''


# Notebook exploring outliers using pyod

In [ ]:
import platform
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CONTAM = 0.05

if "all" in os.getcwd():
    sys.path.append("../..")
    TESTNAME = "SJT"
else:
    sys.path.append(".")
    if len(sys.argv) < 2:
        raise ValueError("Pyod ensemble needs test to run, implemented tests: BAQ, FCA, PAQ, SJT")
    TESTNAME = sys.argv[1].upper()

match TESTNAME:
    case "FCA":
        from func.df_builders.dataframeBuilderFCA import get_df_column_per_combination
    case "SJT":
        from func.df_builders.dataframeBuilderSJT import get_df_column_per_combination
    case "BAQ":
        from func.df_builders.dataframeBuilderBAQ import get_df_column_per_combination
    case "PAQ":
        from func.df_builders.dataframeBuilderPAQ import get_df_column_per_combination

pd.set_option("display.max_rows", 10)


Running pyod ensemble in notebook


In [2]:
from pyod.models.copod import COPOD
from pyod.models.abod import ABOD
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
# from pyod.models.ae1svm import AE1SVM
from pyod.models.knn import KNN
from pyod.models.iforest import IForest
from pyod.models.dif import DIF
from pyod.models.lunar import LUNAR

from pyod.models.pca import PCA as pyodPCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

In [3]:
LOTS_OF_TRAINING_TIME = False # Run all
RUN_OPTIONAL = False # Just skip a few

In [ ]:
def get_normal_dataframe():
    '''
    This function uses the dataframeBuilder of {TESTNAME}, 
    sets the TestKey as index and shuffles the dataset.
    Checks whether CandidateKey is in the dataframe and drops it.
    
    Parameters: None

    Returns:
    a dataframe with the following columns:
        - An answer for each possible question asked
        - Candidate information
        - Amount of days ago
    '''
    df, _, _, _ = get_df_column_per_combination(join_candidate=True)
    df.index = df["TestKey"]
    df.drop(columns="TestKey", inplace=True)
    df = df.sample(frac=1.)

    if "CandidateKey" in df.columns:
        df.drop(columns="CandidateKey", inplace=True)
    return df

In [5]:
df = get_normal_dataframe()
df

,Q0_A1,Q0_A2,Q0_A3,Q1_A1,Q1_A2,Q1_A3,Q2_A1,Q2_A2,Q2_A3,Q3_A1,Q3_A2,Q3_A3,Q4_A1,Q4_A2,Q4_A3,Q5_A1,Q5_A2,Q5_A3,Q6_A1,Q6_A2,Q6_A3,Q7_A1,Q7_A2,Q7_A3,Q8_A1,Q8_A2,Q8_A3,Q9_A1,Q9_A2,Q9_A3,Q10_A1,Q10_A2,Q10_A3,Q11_A1,Q11_A2,Q11_A3,Q12_A1,Q12_A2,Q12_A3,Q13_A1,Q13_A2,Q13_A3,DaysAgo,CandidateKey,VersionNumber_V0.1,VersionNumber_v1.0,InstanceID_1,InstanceID_2,InstanceID_3,InstanceID_5,ChosenLanguage_65CAFEA9-CF49-43ED-99AA-83AFE0539978,ChosenLanguage_BE3C46AE-0192-4A9B-ACEE-37E51836F77C,ChosenLanguage_FADAABC8-26DD-458B-B07E-BF96B4B3FCED,ChosenGender_Gender_Female,ChosenGender_Gender_Male,ChosenGender_Gender_Other,ChosenGender_Gender_Unknown,Gender_Gender_Female,Gender_Gender_Male,Gender_Gender_Unknown,Qualification_Qualification_Bachelor,Qualification_Qualification_MBA,Qualification_Qualification_Master,Qualification_Qualification_PHD,Qualification_Qualification_PostGraduate,Qualification_Qualification_Professional,Qualification_Qualification_Secondary,Qualification_Qualification_Unknown,Qualification_Qualification_Vocational
TestKey,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
166813,0.0,0.0,0.0,1.0,2.0,5.0,2.0,1.0,5.0,5.0,4.0,2.0,1.0,4.0,5.0,5.0,1.0,4.0,5.0,4.0,1.0,1.0,5.0,2.0,2.0,5.0,1.0,5.0,2.0,1.0,1.0,4.0,5.0,5.0,1.0,2.0,5.0,1.0,4.0,5.0,2.0,1.0,367.0,49611021.0,0,1,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0
161075,0.0,0.0,0.0,4.0,2.0,3.0,2.0,3.0,4.0,4.0,2.0,3.0,5.0,1.0,4.0,1.0,3.0,5.0,4.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1360.0,43527771.0,0,1,1,0,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0
166604,0.0,0.0,0.0,5.0,2.0,4.0,4.0,3.0,5.0,4.0,1.0,5.0,5.0,1.0,4.0,4.0,3.0,1.0,5.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,353.0,49656301.0,0,1,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1
162880,0.0,0.0,0.0,5.0,3.0,4.0,3.0,5.0,4.0,5.0,2.0,4.0,5.0,2.0,4.0,1.0,4.0,5.0,5.0,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2593.0,35149841.0,0,1,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0
162009,0.0,0.0,0.0,5.0,2.0,3.0,2.0,1.0,5.0,5.0,1.0,2.0,5.0,3.0,2.0,1.0,2.0,5.0,2.0,3.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2675.0,34583581.0,0,1,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163772,0.0,0.0,0.0,1.0,3.0,5.0,5.0,2.0,4.0,4.0,1.0,5.0,1.0,5.0,4.0,4.0,5.0,2.0,5.0,4.0,1.0,1.0,2.0,5.0,4.0,5.0,1.0,5.0,3.0,4.0,3.0,4.0,5.0,1.0,4.0,5.0,3.0,5.0,4.0,2.0,1.0,5.0,2487.0,35662091.0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
160101,0.0,0.0,0.0,5.0,2.0,3.0,5.0,3.0,4.0,5.0,3.0,1.0,3.0,4.0,5.0,5.0,4.0,3.0,4.0,1.0,2.0,1.0,3.0,5.0,4.0,2.0,5.0,5.0,4.0,3.0,4.0,5.0,1.0,4.0,5.0,2.0,3.0,4.0,5.0,4.0,2.0,1.0,393.0,49463751.0,0,1,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0
166315,0.0,0.0,0.0,3.0,5.0,4.0,1.0,3.0,5.0,4.0,3.0,5.0,5.0,2.0,1.0,1.0,3.0,5.0,4.0,1.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2648.0,34746441.0,0,1,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0


In [ ]:
def data_generator(X, batch_size=1000):
    '''
    This function splits X in batches of size {batch_size}, default 1000, 
    yields a result to be used as an iterator.

    Returns:
        batch_size samples of the dataframe
    '''
    for i in range(0, len(X), batch_size):
        yield X[i:i + batch_size]

# Functie om een model te trainen in batches
def process_batches(model, X, batch_size=1000):
    '''
    Process the fit proces in batches of size {batch_size}, default 1000, 

    Returns:
        Anomaly scores of each test instance in the dataset
    '''
    anomaly_scores = []
    
    # Train op batches
    for batch in data_generator(X, batch_size):
        model.fit(batch)  # Train op de huidige batch
        batch_scores = model.decision_function(batch)  # Verkrijg anomalie scores voor de batch
        anomaly_scores.extend(batch_scores)  # Voeg scores toe aan de lijst
    
    return np.array(anomaly_scores)

def predict_model(model, name, prefix, X_train, contamination, df_is_anomaly):
    '''
    Predicts a certain percentage ({contamination}) of anomalies in the given dataset ({X_train})
    Uses the given model to predict.
    Adds the result in the received dataframe ({df_is_anomaly})

    When the model uses (batched) epochs, .fit and .predict will be called
    otherwise predict_batches

    Parameters:
        model: Pyod model to fit on
        name: used as partial columnname to add is_anomaly results in df_is_anomaly (e.g. lof_default, lof_neigbours10)
        prefix: type of dataframe it is fitted on. e.g. noraml/pca/feature-engineered
        X_train: dataset to fit & predict
        contamination: float in range [0.0, 1.0], percentual amount of outliers the model needs to see as anomaly.
        df_is_anomaly: dataframe containing all predicted results. 

    Returns:
        "Done" : str
    '''
    print(f"Training {name}!")
    
    if "deepforest" in name or "lunar" in name:
        model.fit(X_train)
        anomaly_scores = model.predict(X_train)
    else:
        anomaly_scores = process_batches(model, X_train)

    # Determine anomalies based on scores (e.g., top 5% as anomaly)
    threshold = np.percentile(anomaly_scores, 100 - 100*contamination)  # Drempelwaarde instellen
    is_anomaly = anomaly_scores > threshold
    
    full_name = f"{prefix}_{name}"
    df_is_anomaly[full_name] = is_anomaly.astype(int)

In [ ]:
df_is_anomaly = pd.DataFrame(index=df.index)


In [ ]:
def get_X_train_normal(df):
    '''
    Uses min_max_scales the given dataframe

    Parameters:
        df: X dataframe to make predictions on

    Returns:
        scaled dataframe X_train
    '''
    scaler = MinMaxScaler()
    scaler.fit(df)
    return scaler.transform(df)

def get_X_train_pca(df, n_components=10):
    '''
    Uses Principal Component Analysis on get_X_train_normal

    Parameters:
        df: X dataframe to make predictions on
        n_components (default 10): Amount of dimensions to reduce to when adapting PCA
 
    Returns:
        scaled & pca reduced dimensionality of the dataframe X_train
    '''
    X_train_scaled_normal = get_X_train_normal(df)
    pca = PCA(n_components=n_components)
    return pca.fit_transform(X_train_scaled_normal)


In [9]:
get_X_train_normal(df).shape, get_X_train_pca(df).shape

((6894, 69), (6894, 10))

In [10]:
# This defines all types of dataframes for the pyod ensemble 
# TODO : add pca 95% variance
# PCA(n_components=n_components_95)
# X_pca_95 = pca_95.fit_transform(X_scaled)
dataframes = {
    "normal" : lambda: get_X_train_normal(df),
    "pca-10c" : lambda: get_X_train_pca(df),
}

In [ ]:
# Define models for pyod ensemble
# Some are not worth it to train for FCA
is_fca = True if TESTNAME == "FCA" else False
drop_for_fca = [
    "IForest_2048est_.3sample_.8features",
    "IForest_128est",
    "abod_n_neighors_30",
    "lunar_default",
    "lof_dice",
    "lof_manhattan",
    "lof_32neighbours_leaf64",
]
models = {
    "lof_default" : lambda: LOF(contamination=CONTAM, n_jobs=-1),
    "lof_5neighbours_brute" : lambda: LOF(n_neighbors=5, algorithm='brute', leaf_size=50, metric='minkowski', p=1, metric_params=None, contamination=CONTAM, n_jobs=-1),
    "lof_cosine" : lambda: LOF(n_neighbors=8 if is_fca else 16, algorithm='brute', leaf_size=16, metric='cosine', p=1, metric_params=None, contamination=CONTAM, n_jobs=-1),
    "lof_dice" : lambda: LOF(n_neighbors=8 if is_fca else 16, contamination=CONTAM, metric="dice", n_jobs=-1),
    "lof_32neighbours_leaf64" : lambda: LOF(n_neighbors=32, contamination=CONTAM, leaf_size=32 if is_fca else 64, n_jobs=-1),
    "lof_manhattan" : lambda: LOF(metric='manhattan', contamination=CONTAM, n_jobs=-1),

    "IForest_default" : lambda: IForest(contamination=CONTAM, n_jobs=-1),
    "IForest_128est" : lambda: IForest(contamination=CONTAM, n_jobs=-1, n_estimators=128),
    "IForest_256est_bootstrap" : lambda: IForest(contamination=CONTAM, n_jobs=-1, n_estimators=256, bootstrap=True),
    "IForest_256est_.7features" : lambda: IForest(contamination=CONTAM, n_jobs=-1, n_estimators=256, max_features=0.7),
    "IForest_512est_.77samp" : lambda: IForest(contamination=CONTAM, n_estimators=555, max_samples=0.77, max_features=0.55, bootstrap=True, n_jobs=-1),
    "IForest_1024est_.44sample" : lambda: IForest(contamination=CONTAM, n_jobs=-1, n_estimators=1024, max_samples=0.44),
    "IForest_2048est_.3sample_.8features" : lambda: IForest(contamination=CONTAM, n_jobs=-1, n_estimators=2024, max_samples=0.3, max_features=0.8),

    "knn_default" : lambda: KNN(contamination=CONTAM, n_jobs=-1),
    "knn_mean" : lambda: KNN(contamination=CONTAM, method="mean", n_jobs=-1),
    "knn_16neighbours" : lambda: KNN(contamination=CONTAM, n_neighbors=8 if is_fca else 16, method="mean", n_jobs=-1),

    "deepforest_default" : lambda: DIF(contamination=CONTAM),
    "deepforest_params" :lambda: DIF(batch_size=1024, hidden_neurons=[64,64,32], hidden_activation='relu', skip_connection=True, n_ensemble=32, n_estimators=8, max_samples=128, contamination=0.1, random_state=None, device=None),
    
    "lunar_default" : lambda: LUNAR(contamination=CONTAM),
    "lunar_params": lambda: LUNAR(n_neighbours=4, val_size=0.1, proportion=0.8, n_epochs=150, lr=0.0011, contamination=CONTAM),

    "abod_default" : lambda: ABOD(contamination=CONTAM),
    "abod_n_neighors_10" : lambda: ABOD(contamination=CONTAM, n_neighbors=10),
    "abod_n_neighors_30" : lambda: ABOD(contamination=CONTAM, n_neighbors=30),

    "pca_1" : lambda: pyodPCA(n_components=0.95, svd_solver="full", contamination=CONTAM),
    "pca_max_components" : lambda: pyodPCA(n_components=min(len(df.columns), 128), svd_solver="full", contamination=CONTAM),
}
if TESTNAME == "FCA":
    for m in drop_for_fca:
        del models[m]
models


{'lof_default': <function __main__.<lambda>()>,
 'lof_5neighbours_brute': <function __main__.<lambda>()>,
 'lof_cosine': <function __main__.<lambda>()>,
 'lof_dice': <function __main__.<lambda>()>,
 'lof_32neighbours_leaf64': <function __main__.<lambda>()>,
 'lof_manhattan': <function __main__.<lambda>()>,
 'IForest_default': <function __main__.<lambda>()>,
 'IForest_128est': <function __main__.<lambda>()>,
 'IForest_256est_bootstrap': <function __main__.<lambda>()>,
 'IForest_256est_.7features': <function __main__.<lambda>()>,
 'IForest_512est_.77samp': <function __main__.<lambda>()>,
 'IForest_1024est_.44sample': <function __main__.<lambda>()>,
 'IForest_2048est_.3sample_.8features': <function __main__.<lambda>()>,
 'knn_default': <function __main__.<lambda>()>,
 'knn_mean': <function __main__.<lambda>()>,
 'knn_16neighbours': <function __main__.<lambda>()>,
 'deepforest_default': <function __main__.<lambda>()>,
 'deepforest_params': <function __main__.<lambda>()>,
 'lunar_default': 

In [ ]:
# Iterate one type of dataframe at a time, to preserve memory usage
# Then fit all the defined models above
for prefix, df_function in dataframes.items():
    X_train = df_function()
    print(f"Fitting {prefix} dataframe with shape: {X_train.shape}")
    # Iterate all models
    for modelname, modelfunction in models.items():
        model = modelfunction()
        start = time.time()
        try:
            predict_model(
                modelfunction(),
                name=f"{modelname}",
                prefix=prefix,
                X_train=X_train, 
                contamination=CONTAM, 
                df_is_anomaly=df_is_anomaly,
            )
        except Exception as e:
            print(f"An error occurred while executing the prediction of {modelname}: {str(e)}")
        end = time.time()
        seconds = end-start
        print(f"{modelname} took {seconds:.0f}s")
    print("="*70)

Fitting normal dataframe with shape: (6894, 69)
Training lof_default! Took 0s
Training lof_5neighbours_brute! Took 0s
Training lof_cosine! Took 1s
Training lof_dice! 

/home/miked/code/dep2-g2/venv/lib/python3.12/site-packages/sklearn/metrics/pairwise.py:2361: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.warn(msg, DataConversionWarning)
/home/miked/code/dep2-g2/venv/lib/python3.12/site-packages/sklearn/metrics/pairwise.py:2361: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.warn(msg, DataConversionWarning)
/home/miked/code/dep2-g2/venv/lib/python3.12/site-packages/sklearn/metrics/pairwise.py:2361: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.warn(msg, DataConversionWarning)
/home/miked/code/dep2-g2/venv/lib/python3.12/site-packages/sklearn/metrics/pairwise.py:2361: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.warn(msg, DataConversionWarning)
/home/miked/code/dep2-g2/venv/lib/python3.12/site-packages/sklearn/metrics/pairwise.py:2361: DataConversionWarning: Data was converted to boolean for metric dice
  warnings.war

Took 1s
Training lof_32neighbours_leaf64! Took 0s
Training lof_manhattan! Took 0s
Training IForest_default! Took 2s
Training IForest_128est! Took 2s
Training IForest_256est_bootstrap! Took 3s
Training IForest_256est_.7features! Took 4s
Training IForest_512est_.77samp! Took 10s
Training IForest_1024est_.44sample! Took 17s
Training IForest_2048est_.3sample_.8features! Took 37s
Training knn_default! Took 0s
Training knn_mean! Took 1s
Training knn_16neighbours! Took 1s
Training deepforest_default! Took 17s
Training deepforest_params! Took 9s
Training lunar_default! Took 19s
Training lunar_params! Took 12s
Training abod_default! Took 2s
Training abod_n_neighors_10! Took 3s
Training abod_n_neighors_30! Took 27s
Training pca_1! Took 0s
Training pca_max_components! Took 0s
Fitting pca-10c dataframe with shape: (6894, 10)
Training lof_default! Took 0s
Training lof_5neighbours_brute! Took 0s
Training lof_cosine! Took 0s
Training lof_dice! Took 1s
Training lof_32neighbours_leaf64! Took 0s
Trainin

In [13]:
df_is_anomaly

,normal_lof_default,normal_lof_5neighbours_brute,normal_lof_cosine,normal_lof_dice,normal_lof_32neighbours_leaf64,normal_lof_manhattan,normal_IForest_default,normal_IForest_128est,normal_IForest_256est_bootstrap,normal_IForest_256est_.7features,normal_IForest_512est_.77samp,normal_IForest_1024est_.44sample,normal_IForest_2048est_.3sample_.8features,normal_knn_default,normal_knn_mean,normal_knn_16neighbours,normal_deepforest_default,normal_deepforest_params,normal_lunar_default,normal_lunar_params,normal_abod_default,normal_abod_n_neighors_10,normal_abod_n_neighors_30,normal_pca_1,normal_pca_max_components,pca-10c_lof_default,pca-10c_lof_5neighbours_brute,pca-10c_lof_cosine,pca-10c_lof_dice,pca-10c_lof_32neighbours_leaf64,pca-10c_lof_manhattan,pca-10c_IForest_default,pca-10c_IForest_128est,pca-10c_IForest_256est_bootstrap,pca-10c_IForest_256est_.7features,pca-10c_IForest_512est_.77samp,pca-10c_IForest_1024est_.44sample,pca-10c_IForest_2048est_.3sample_.8features,pca-10c_knn_default,pca-10c_knn_mean,pca-10c_knn_16neighbours,pca-10c_deepforest_default,pca-10c_deepforest_params,pca-10c_lunar_default,pca-10c_lunar_params,pca-10c_abod_default,pca-10c_abod_n_neighors_10,pca-10c_abod_n_neighors_30,pca-10c_pca_1
TestKey,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
166813,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,1,0,1,1,0,0,1,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0
161075,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
166604,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
162880,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
162009,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163772,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,1,1,0,1,1,0,0,0,0,0,0,1,0,1,1,1,1,1,1,1,1,0,1,0,0,0,1,0,0,0,1
160101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
166315,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
def compare_models_individual(df, threshold=97):
    '''
    Calculate the similarity between each model result in df.columns.
    Prints similar model results > {threshold}

    Parameters:
        df: dataframe with anomaly results
        threshold: default 97, when to print if a model predicts an equal result
    '''
    columns = df.columns.to_list()
    columns_copy = columns.copy()
    for col in columns:
        columns_copy.remove(col)
        for other in columns_copy:
            similarity = (df[col] == df[other]).sum() / len(df) * 100
            # if similarity >= threshold:
            #     print(f"{similarity:.2f}% = similarity between {col} & {other} ")

# compare_models_individual(df_is_anomaly)

In [ ]:
def compare_models_total(df):
    '''
    Calculate the similarity between each model result in df.columns.
        ! Watch out ! df.columns * df.columns results!
        Also compares with itself --> 100% similarity.
        Similarity between 0-100%

    Parameters:
        df: dataframe with anomaly results

    Returns:
        A dataframe with each comparison, with columns:
            - df: df type1 (e.g. normal/pca/feature-engineered)
            - other_df: df_type2 (compared to)
            - model: pyod model (e.g. LOF, IForest, ABOD...)
            - other_model: pyod model which is compared to
            - variant: parameter description
            - other variant: parameter description of the other model
    '''
    columns = df.columns.to_list()
    columns_copy = columns.copy()
    individual_simularities = {
        "df" : [],
        "model" : [],
        "variant": [],
        "other_var" : [],
        "other_model" : [],
        "other_df" : [],
        "similarity": [],
    }
    total_similarity = {col:0 for col in columns_copy}
    for col in columns:
        for other in columns_copy:
            # if other != col:
            # Yes compare with itself, but because everyone is compared with itself, 
            # later functions add +100 for every, still keeping the lowest similirity between the same models
            similarity = (df[col] == df[other]).sum() / len(df) * 100
            total_similarity[col] += similarity
            splitted_col = col.split("_")
            splitted_other = other.split("_")
            individual_simularities["df"].append(splitted_col[0])
            individual_simularities["model"].append(splitted_col[1])
            individual_simularities["variant"].append("_".join(splitted_col[2:]))
            individual_simularities["other_df"].append(splitted_other[0])
            individual_simularities["other_model"].append(splitted_other[1])
            individual_simularities["other_var"].append("_".join(splitted_other[2:]))
            individual_simularities["similarity"].append(similarity)

    return pd.DataFrame(individual_simularities)

df_compared = compare_models_total(df_is_anomaly)
df_compared

,df,model,variant,other_var,other_model,other_df,similarity
0,normal,lof,default,default,lof,normal,100.000000
1,normal,lof,default,5neighbours_brute,lof,normal,95.619379
2,normal,lof,default,cosine,lof,normal,98.810560
3,normal,lof,default,dice,lof,normal,95.358283
4,normal,lof,default,32neighbours_leaf64,lof,normal,96.895851
...,...,...,...,...,...,...,...
2396,pca-10c,pca,1,params,lunar,pca-10c,93.414563
2397,pca-10c,pca,1,default,abod,pca-10c,93.211488
2398,pca-10c,pca,1,n_neighors_10,abod,pca-10c,93.762692
2399,pca-10c,pca,1,n_neighors_30,abod,pca-10c,94.052800


In [ ]:
# Compare models of the same type, e.g. IForest, LOF to keep those 3 who are less similar with each other.
# Splitted in two notebook cells to show an inbetween result
tmp = df_compared[(df_compared["model"] == df_compared["other_model"]) & (df_compared["df"] == df_compared["other_df"])]
tmp = tmp.groupby(["df", "model", "variant"]).sum("similarity").sort_values("similarity", ascending=False).reset_index()
tmp

,df,model,variant,similarity
0,normal,IForest,2048est_.3sample_.8features,690.600522
1,normal,IForest,1024est_.44sample,690.281404
2,normal,IForest,512est_.77samp,689.788222
3,normal,IForest,256est_.7features,688.569771
4,normal,IForest,256est_bootstrap,688.105599
...,...,...,...,...
44,normal,pca,max_components,195.155207
45,normal,pca,1,195.155207
46,normal,deepforest,params,194.995648
47,normal,deepforest,default,194.995648


In [ ]:
tmp = tmp.groupby(["df", "model"]).tail(3)

filtered_columns = (tmp["df"] + "_" + tmp["model"] + "_" + tmp["variant"])
filtered_columns

4      normal_IForest_256est_bootstrap
7               normal_IForest_default
8                normal_IForest_128est
11    pca-10c_IForest_256est_bootstrap
12              pca-10c_IForest_128est
                    ...               
44           normal_pca_max_components
45                        normal_pca_1
46            normal_deepforest_params
47           normal_deepforest_default
48                       pca-10c_pca_1
Length: 35, dtype: object

In [ ]:
# Filter anomaly datframe to keep only the wanted models
df_is_anomaly = df_is_anomaly[filtered_columns].copy()
df_is_anomaly

,normal_IForest_256est_bootstrap,normal_IForest_default,normal_IForest_128est,pca-10c_IForest_256est_bootstrap,pca-10c_IForest_128est,pca-10c_IForest_default,normal_lof_32neighbours_leaf64,normal_lof_5neighbours_brute,pca-10c_lof_5neighbours_brute,pca-10c_lof_dice,normal_lof_dice,pca-10c_lof_32neighbours_leaf64,normal_knn_default,normal_knn_mean,normal_knn_16neighbours,pca-10c_knn_default,pca-10c_knn_mean,pca-10c_knn_16neighbours,pca-10c_abod_n_neighors_10,pca-10c_abod_n_neighors_30,pca-10c_abod_default,normal_abod_n_neighors_10,normal_abod_n_neighors_30,normal_abod_default,pca-10c_deepforest_params,pca-10c_deepforest_default,pca-10c_lunar_default,pca-10c_lunar_params,normal_lunar_params,normal_lunar_default,normal_pca_max_components,normal_pca_1,normal_deepforest_params,normal_deepforest_default,pca-10c_pca_1
TestKey,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
166813,1,1,1,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,1,0,1,1,1,0,0,0,0,0,1,0,0,0,0,0,0
161075,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
166604,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
162880,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
162009,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163772,0,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,0,1,0,0,0,1,1,0,0,0,0,1,1,1,0,0,0,0,1
160101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
166315,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Print/show the amount of kept variations for each model
# Divide by the amount of dataframes, to as otherwise its counted N times for each dataframe type
tmp.groupby(["model"]).count() / len(dataframes)

,df,variant,similarity
model,,,
IForest,3.0,3.0,3.0
abod,3.0,3.0,3.0
deepforest,2.0,2.0,2.0
knn,3.0,3.0,3.0
lof,3.0,3.0,3.0
lunar,2.0,2.0,2.0
pca,1.5,1.5,1.5


In [ ]:
# Calculate the percentage of models that predicted a given test as anomaly
# Use count as inbetween result
columns = df_is_anomaly.columns.to_list()
df_is_anomaly.loc[:, 'count'] =  df_is_anomaly[columns].sum(axis=1)
df_is_anomaly.loc[:, 'percentage'] = df_is_anomaly['count'] / len(columns)
df_is_anomaly

,normal_IForest_256est_bootstrap,normal_IForest_default,normal_IForest_128est,pca-10c_IForest_256est_bootstrap,pca-10c_IForest_128est,pca-10c_IForest_default,normal_lof_32neighbours_leaf64,normal_lof_5neighbours_brute,pca-10c_lof_5neighbours_brute,pca-10c_lof_dice,normal_lof_dice,pca-10c_lof_32neighbours_leaf64,normal_knn_default,normal_knn_mean,normal_knn_16neighbours,pca-10c_knn_default,pca-10c_knn_mean,pca-10c_knn_16neighbours,pca-10c_abod_n_neighors_10,pca-10c_abod_n_neighors_30,pca-10c_abod_default,normal_abod_n_neighors_10,normal_abod_n_neighors_30,normal_abod_default,pca-10c_deepforest_params,pca-10c_deepforest_default,pca-10c_lunar_default,pca-10c_lunar_params,normal_lunar_params,normal_lunar_default,normal_pca_max_components,normal_pca_1,normal_deepforest_params,normal_deepforest_default,pca-10c_pca_1,count,percentage
TestKey,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
166813,1,1,1,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,1,0,1,1,1,0,0,0,0,0,1,0,0,0,0,0,0,14,0.400000
161075,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0.057143
166604,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0.028571
162880,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000
162009,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163772,0,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,0,1,0,0,0,1,1,0,0,0,0,1,1,1,0,0,0,0,1,15,0.428571
160101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000
166315,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000


In [21]:
df_is_anomaly['count'].value_counts()

count
0     4780
1      766
2      341
3      210
4      139
      ... 
31       6
29       4
30       3
28       1
33       1
Name: count, Length: 33, dtype: int64

In [ ]:
# Based on THRESHOLD, decide which test is considered an anomaly
TRESHOLD = 0.5
df_is_anomaly["is_anomaly"] = np.where(df_is_anomaly['percentage'] >= TRESHOLD, 1, 0)
df_is_anomaly

,normal_IForest_256est_bootstrap,normal_IForest_default,normal_IForest_128est,pca-10c_IForest_256est_bootstrap,pca-10c_IForest_128est,pca-10c_IForest_default,normal_lof_32neighbours_leaf64,normal_lof_5neighbours_brute,pca-10c_lof_5neighbours_brute,pca-10c_lof_dice,normal_lof_dice,pca-10c_lof_32neighbours_leaf64,normal_knn_default,normal_knn_mean,normal_knn_16neighbours,pca-10c_knn_default,pca-10c_knn_mean,pca-10c_knn_16neighbours,pca-10c_abod_n_neighors_10,pca-10c_abod_n_neighors_30,pca-10c_abod_default,normal_abod_n_neighors_10,normal_abod_n_neighors_30,normal_abod_default,pca-10c_deepforest_params,pca-10c_deepforest_default,pca-10c_lunar_default,pca-10c_lunar_params,normal_lunar_params,normal_lunar_default,normal_pca_max_components,normal_pca_1,normal_deepforest_params,normal_deepforest_default,pca-10c_pca_1,count,percentage,is_anomaly
TestKey,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
166813,1,1,1,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,1,0,1,1,1,0,0,0,0,0,1,0,0,0,0,0,0,14,0.400000,0
161075,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0.057143,0
166604,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0.028571,0
162880,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0
162009,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163772,0,0,0,1,1,1,0,0,0,0,0,1,1,1,1,1,0,1,0,0,0,1,1,0,0,0,0,1,1,1,0,0,0,0,1,15,0.428571,0
160101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0
166315,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0


In [23]:
df_is_anomaly["is_anomaly"].sum()

np.int64(143)

In [ ]:
# Write result as csv
anomalies_path = os.path.abspath("anomalies")
csv_path = os.path.join(anomalies_path, TESTNAME, "csv", "pyod_ensemble.csv")
df_is_anomaly.to_csv(csv_path, index=True)


OSError: Cannot save file into a non-existent directory: '/home/miked/code/dep2-g2/anomalies/all/anomalies/SJT/csv'